# DATA EXPLORATION - VANGUARD A/B TEST
---



**Question was: Would these changes encourage more clients to complete the process?**
Database Client Profiles (df_final_demo): Demographics like age, gender, and account details of our clients.
Experiment Roster (df_final_experiment_clients): A list revealing which clients were part of the grand experiment.

IMPORT LIBRARIES

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

sns.set_style("whitegrid")


1. LOAD THE DATASETS 

In [2]:
data_path = "data/raw_data"
df_demo = pd.read_csv(f"{data_path}/df_final_demo.txt")
df_experiment = pd.read_csv(f"{data_path}/df_final_experiment_clients.txt")

## Phase 1 — EDA & Data Cleaning

DATASET: DEMOGRAPHICS

In [3]:
df_demo.head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0


In [4]:
df_demo.dtypes

client_id             int64
clnt_tenure_yr      float64
clnt_tenure_mnth    float64
clnt_age            float64
gendr                object
num_accts           float64
bal                 float64
calls_6_mnth        float64
logons_6_mnth       float64
dtype: object

In [5]:
df_demo.shape

(70609, 9)

DATASET: EXPERIMENT ROSTER

In [6]:
df_experiment.head()

,client_id,Variation
0,9988021,Test
1,8320017,Test
2,4033851,Control
3,1982004,Test
4,9294070,Control


In [7]:
df_experiment.shape

(70609, 2)

In [8]:
# merging both databases

df_exde = pd.merge(df_demo, df_experiment, on='client_id', how='outer')
df_exde

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation
0,169,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN
1,555,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test
2,647,12.0,151.0,57.5,M,2.0,30525.80,0.0,4.0,Test
3,722,11.0,143.0,59.5,F,2.0,22466.17,1.0,1.0,NaN
4,934,9.0,109.0,51.0,F,2.0,32522.88,0.0,3.0,Test
...,...,...,...,...,...,...,...,...,...,...
70604,9999400,7.0,86.0,28.5,U,2.0,51787.04,0.0,3.0,Test
70605,9999626,9.0,113.0,35.0,M,2.0,36642.88,6.0,9.0,Test
70606,9999729,10.0,124.0,31.0,F,3.0,107059.74,6.0,9.0,Test
70607,9999832,23.0,281.0,49.0,F,2.0,431887.61,1.0,4.0,Test


**Rename Columns**

In [9]:
#renaming columns
df_exde.columns = df_exde.columns.str.lower().str.replace(' ', '_')
df_exde.rename(columns= {'gendr': 'gender', "bal": 'balance'}, inplace=True)


In [10]:
df_exde.head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gender,num_accts,balance,calls_6_mnth,logons_6_mnth,variation
0,169,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN
1,555,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test
2,647,12.0,151.0,57.5,M,2.0,30525.80,0.0,4.0,Test
3,722,11.0,143.0,59.5,F,2.0,22466.17,1.0,1.0,NaN
4,934,9.0,109.0,51.0,F,2.0,32522.88,0.0,3.0,Test


In [11]:
df_exde.dtypes

client_id             int64
clnt_tenure_yr      float64
clnt_tenure_mnth    float64
clnt_age            float64
gender               object
num_accts           float64
balance             float64
calls_6_mnth        float64
logons_6_mnth       float64
variation            object
dtype: object

## Dealing with NULL VALUES

In [12]:
# finding nulls
df_exde.isna().sum()

client_id               0
clnt_tenure_yr         14
clnt_tenure_mnth       14
clnt_age               15
gender                 14
num_accts              14
balance                14
calls_6_mnth           14
logons_6_mnth          14
variation           20109
dtype: int64

In [13]:
df_exde.dropna(subset=['variation'], inplace=True)

In [14]:
# finding nulls
df_exde.isna().sum()

client_id            0
clnt_tenure_yr      12
clnt_tenure_mnth    12
clnt_age            13
gender              12
num_accts           12
balance             12
calls_6_mnth        12
logons_6_mnth       12
variation            0
dtype: int64

In [15]:
num_cols = df_exde.select_dtypes(include='number').columns

df_exde[num_cols] = df_exde[num_cols].fillna(df_exde[num_cols].median())

In [16]:
df_exde["gender"].value_counts()

gender
U    17280
M    16947
F    16259
X        2
Name: count, dtype: int64

In [17]:
# formating a column with a function
def clean_sex_column(x):
    if pd.isna(x):
        return "unknown"
    if x in ["F"]:
        return "female"
    elif x in ["M"]:
        return "male"
    elif x in ["U", "X"]:
        return "unknown"
    else:
        return "unknown"

In [18]:
df_exde['gender'] = df_exde['gender'].apply(clean_sex_column) 

In [19]:
# finding nulls
df_exde.isna().sum()

client_id           0
clnt_tenure_yr      0
clnt_tenure_mnth    0
clnt_age            0
gender              0
num_accts           0
balance             0
calls_6_mnth        0
logons_6_mnth       0
variation           0
dtype: int64

In [20]:
# duplicates sum
df_exde.duplicated().sum()

np.int64(0)

## Descriptive Analysis Numerical Variables

In [21]:
# descriptive analysis numerical variables
pd.set_option('display.float_format', '{:.2f}'.format)
df_exde.describe()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,num_accts,balance,calls_6_mnth,logons_6_mnth
count,50500.00,50500.00,50500.00,50500.00,50500.00,50500.00,50500.00,50500.00
mean,5006179.06,12.03,150.41,47.32,2.25,149494.77,3.09,6.13
std,2877442.53,6.86,81.94,15.52,0.53,302003.29,2.19,2.18
min,555.00,2.00,33.00,17.00,1.00,23789.44,0.00,3.00
25%,2515645.75,6.00,82.00,33.50,2.00,39881.04,1.00,4.00
50%,5025103.50,11.00,136.00,48.00,2.00,65733.60,3.00,6.00
75%,7477933.25,16.00,192.00,59.50,2.00,139938.95,5.00,8.00
max,9999832.00,55.00,669.00,96.00,7.00,16320040.15,6.00,9.00


## Descriptive Analysis Categorical Variables

In [22]:
# descriptive analysis categorical variables 
df_exde["variation"].value_counts()

variation
Test       26968
Control    23532
Name: count, dtype: int64

In [23]:
df_exde["gender"].value_counts()

gender
unknown    17294
male       16947
female     16259
Name: count, dtype: int64

In [24]:
df_exde.head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gender,num_accts,balance,calls_6_mnth,logons_6_mnth,variation
1,555,3.00,46.00,29.50,unknown,2.00,25454.66,2.00,6.00,Test
2,647,12.00,151.00,57.50,male,2.00,30525.80,0.00,4.00,Test
4,934,9.00,109.00,51.00,female,2.00,32522.88,0.00,3.00,Test
5,1028,12.00,145.00,36.00,male,3.00,103520.22,1.00,4.00,Control
6,1104,5.00,66.00,48.00,unknown,3.00,154643.94,6.00,9.00,Control


## Exporting Database

In [25]:
# exporting database
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

df_exde.to_csv(f"{output_dir}/df_exde.csv", index=False)

print("Exported:")
for f in os.listdir(output_dir):
    path = os.path.join(output_dir, f)
    size = os.path.getsize(path)
    print(f"  {f} ({size:,} bytes)")

Exported:
  web.csv (40,150,127 bytes)
  df_exde.csv (2,986,562 bytes)
